In [1]:
# going to begin import necessary packages
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt 
# importing the necessary packages for the model 
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
# for the first model going to attempt to do a resnet18 model 
from torchvision.models import resnet18, ResNet18_Weights
from torch.nn import Module
from PIL import Image
from io import BytesIO
import requests
from torch.utils.data import DataLoader, Dataset, random_split
from torch import Generator
import torch

I0000 00:00:1775837023.467745  544665 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [3]:
# now going to read in our created dataframe 
df = pd.read_csv('/home/ybf3jw/ds 6050 project/final_df.csv')

In [4]:
df.head()

,Unnamed: 0,filename,link,id,gender,masterCategory,subCategory,articleType,baseColour,season,year,usage,productDisplayName,image_data
0,0,1638,http://assets.myntassets.com/v1/images/style/p...,1638,Unisex,Apparel,Bottomwear,Swimwear,Blue,Fall,2010.0,Sports,Nabaiji Swimming Goggles Blue Black,<PIL.Image.Image image mode=RGB size=1800x2400...
1,1,32903,http://assets.myntassets.com/v1/images/style/p...,32903,Women,Apparel,Bottomwear,Shorts,Red,Summer,2012.0,Casual,Allen Solly Women Red Shorts,<PIL.Image.Image image mode=RGB size=1080x1440...
2,2,39381,http://assets.myntassets.com/v1/images/style/p...,39381,Men,Apparel,Bottomwear,Jeans,Black,Summer,2012.0,Casual,Peter England Men Party Black Jeans,<PIL.Image.Image image mode=RGB size=1800x2400...
3,3,12163,http://assets.myntassets.com/v1/images/style/p...,12163,Women,Apparel,Bottomwear,Leggings,Purple,Fall,2011.0,Ethnic,Aurelia Women Solid Purple Leggings,<PIL.Image.Image image mode=RGB size=1800x2400...
4,4,1607,http://assets.myntassets.com/v1/images/style/p...,1607,Men,Apparel,Bottomwear,Track Pants,Blue,Fall,2010.0,Sports,Reebok Men trackpant- male Track Pants,<PIL.Image.Image image mode=RGB size=1080x1440...


In [5]:
df['subCategory'].value_counts()

subCategory
Bottomwear    500
Topwear       500
Shoes         500
Name: count, dtype: int64

In [6]:
df['season'].value_counts()

season
Summer    815
Fall      500
Winter    158
Spring     26
Name: count, dtype: int64

In [7]:
df['gender'].value_counts()

gender
Men       834
Women     553
Boys       43
Girls      41
Unisex     29
Name: count, dtype: int64

In [8]:
# going to create a new label for the CNN; the label will consist of gender, season, and subcategory!
pd.set_option('display.max_rows', None)
df['img_label'] = df['subCategory'] + ' ' + df['season'] + ' ' + df['gender']
df['img_label'].value_counts()

img_label
Topwear Summer Men         158
Shoes Summer Men           141
Bottomwear Summer Men      129
Bottomwear Summer Women    126
Shoes Fall Men             115
Topwear Summer Women       112
Topwear Fall Men           112
Bottomwear Fall Men        110
Shoes Winter Women          97
Bottomwear Fall Women       68
Topwear Fall Women          65
Shoes Summer Women          52
Shoes Winter Men            35
Bottomwear Summer Girls     24
Topwear Summer Boys         20
Bottomwear Summer Boys      19
Shoes Fall Women            18
Topwear Summer Girls        16
Shoes Summer Unisex         13
Shoes Spring Men            10
Topwear Winter Men           9
Bottomwear Winter Women      8
Bottomwear Winter Men        7
Shoes Spring Women           7
Bottomwear Spring Men        6
Shoes Fall Unisex            6
Topwear Summer Unisex        5
Bottomwear Fall Unisex       2
Topwear Fall Boys            2
Shoes Spring Unisex          2
Bottomwear Fall Girls        1
Topwear Spring Men           

In [9]:
class ImageDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

        # convert labels to numeric
        self.label_map = {label: i for i, label in enumerate(df['img_label'].unique())}
        self.df['label'] = self.df['img_label'].map(self.label_map)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        url = self.df.iloc[idx]['link']
        label = self.df.iloc[idx]['label']
    

        try:
            response = requests.get(url, timeout=5)
            img = Image.open(BytesIO(response.content)).convert('RGB')
        except:
            return self.__getitem__((idx + 1) % len(self.df))

        if self.transform:
            img = self.transform(img)

        return img, label

In [10]:
from torchvision import transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [11]:
# need to get the number of classes in the in the articletype clothing 
num_of_classes = df['img_label'].nunique()
print(num_of_classes)

35


In [12]:
dataset = ImageDataset(df, transform=transform)
print(dataset.df['label'].min())
print(dataset.df['label'].max())
# copying code from assignment 2 
SEED = 42
TEST_RATIO = 0.2
size_all = len(dataset)
size_test = int(size_all * TEST_RATIO)
size_train = size_all - size_test

dataset_train, dataset_test = random_split(dataset, [size_train, size_test], generator=Generator().manual_seed(SEED))

0
35


In [13]:
# setting up the parameters for the ResNet structutre
batch_size = 32
epochs = 200
data_augmentation = True
subtract_pixel_mean = True

import torch.nn as nn

class MyCNN1(Module):
    def __init__(self, num_of_classes):
        super().__init__()
        weights = ResNet18_Weights.DEFAULT # going to use the default resnet weights
        self.resnet18 = resnet18(weights=weights)
        self.resnet18.fc = nn.Linear(
            self.resnet18.fc.in_features,
            num_of_classes
        )
        self.transforms = weights.transforms(antialias=True)

    def forward(self, X):
        # X = self.transforms(X)
        # y_pred = self.resnet18(X)
        # return y_pred
        return self.resnet18(X)

In [14]:
model1 = MyCNN1(num_of_classes).to('cuda') 
model1.to('cuda')
print(model1) # okay yay the model is good 

MyCNN1(
  (resnet18): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_run

_______

In [15]:
# now that the data has been split and the resnet model has been built need to feed the images into the model
# first need to import the important packages to train and look at the accuracy 
from torch.nn import Sequential
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model1.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.1)

In [17]:
# now going to train the model 
num_epochs = 1
train_losses, train_acc_list, test_losses, test_acc_list = [], [], [], []
trainloader = torch.utils.data.DataLoader(dataset_train, batch_size=16, shuffle=True, num_workers=1)
testloader = torch.utils.data.DataLoader(dataset_test, batch_size=16, shuffle=False, num_workers=1)

data_iter = iter(trainloader)
images, labels = next(data_iter)

print(images.shape)
print(labels.shape)
print(labels.dtype)
print(labels.min(), labels.max())

for epoch in range(num_epochs):
    model1.train()
    running_loss = 0.0
    correct, total = 0, 0
    for inputs, labels in trainloader:
        inputs, labels = inputs.to('cuda'), labels.to('cuda')
        if labels.ndim > 1:
          labels = labels.argmax(dim=1)
        labels = labels.long()
        optimizer.zero_grad()
        outputs = model1(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    avg_train_loss = running_loss / len(trainloader)
    train_losses.append(avg_train_loss)

    with torch.no_grad():
      correct = 0
      total_loss = 0
      size = 0
      for inputs, labels in testloader:
        inputs, labels = inputs.to('cuda'), labels.to('cuda')
        if labels.ndim > 1:
          labels = labels.argmax(dim=1)
        labels = labels.long()
        outputs = model1(inputs)
        print("outputs shape:", outputs.shape)
        print("labels shape:", labels.shape)
        print("labels min/max:", labels.min().item(), labels.max().item())
        print("labels dtype:", labels.dtype)
        assert labels.dtype == torch.long
        assert labels.min() >= 0
        assert labels.max() < num_of_classes
        loss = criterion(outputs,labels)
        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).float().sum().item()
        size += labels.size(0)
      avg_test_loss = total_loss / len(testloader)
      test_losses.append(avg_test_loss)
    accuracy = correct / size

    print(f"Test Error:\n Accuracy: {(100 * accuracy):>0.1f}%\n")

torch.Size([16, 3, 224, 224])
torch.Size([16])
torch.int64
tensor(1) tensor(25)


AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
print(train_losses)
print(test_losses)

In [ ]:
# lets see if i can make a loss plot


plt.plot(train_losses, label='Train Loss')
plt.plot(test_losses, label='Test Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Test Loss')
plt.legend()
plt.show()

In [ ]:
# need to create an AUC visualization